In [ ]:
import numpy as np
import hashlib
import scipy.sparse as sp
from datasketch import MinHash, MinHashLSH

# Function to load user-movie rating data from a .npy file
def load_data(file_path):
    try:
        data = np.load(file_path)  # Load the .npy file
        print(f"Data loaded successfully with shape {data.shape}")
        return data
    except Exception as e:
        print(f"Error loading data: {e}")
        return None

# Create sparse user-movie matrix (using COO format)
def create_sparse_matrix(data):
    num_users = np.max(data[:, 0])  # Maximum user id
    num_movies = np.max(data[:, 1])  # Maximum movie id
    print(f"Number of users: {num_users}, Number of movies: {num_movies}")
    # Create the sparse matrix in COO format
    user_movie_matrix = sp.coo_matrix((np.ones(len(data)), (data[:, 0] - 1, data[:, 1] - 1)),
                                      shape=(num_users, num_movies))
    print(f"Sparse matrix shape: {user_movie_matrix.shape}")
    return user_movie_matrix

# Murmur hash function
def murmur_hash(seed, value):
    return int(hashlib.md5(f'{seed}{value}'.encode('utf-8')).hexdigest(), 16)

# Minhashing: Generate signature matrix using hash functions (Murmur Hash)
def minhash(user_movie_matrix, num_hashes):
    num_users, num_movies = user_movie_matrix.shape
    signature_matrix = np.full((num_hashes, num_users), np.inf)

    for user_id in range(num_users):
        user_ratings = user_movie_matrix.getrow(user_id).toarray().flatten()
        for i in range(num_hashes):
            min_hash = np.inf
            for movie_id in range(num_movies):
                if user_ratings[movie_id] == 1:  # User has rated this movie
                    hash_value = murmur_hash(i, movie_id)
                    min_hash = min(min_hash, hash_value)
            signature_matrix[i, user_id] = min_hash
    return signature_matrix

# Locality-Sensitive Hashing (LSH) using datasketch for banding
def lsh_banding(signature_matrix, num_bands, rows_per_band, threshold=0.5):
    num_hashes, num_users = signature_matrix.shape
    assert num_bands * rows_per_band == num_hashes, \
        "Number of bands * rows per band must equal total hash functions"
    
    lsh = MinHashLSH(threshold=threshold)
    candidate_pairs = set()

    # Process each band
    for band_idx in range(num_bands):
        # Extract rows corresponding to the current band
        start_row = band_idx * rows_per_band
        end_row = start_row + rows_per_band
        band = signature_matrix[start_row:end_row, :]

        # Hash columns (users) in the band into buckets
        buckets = {}
        for user_idx in range(num_users):
            # Use the band as the key for hashing
            band_hash = hash(tuple(band[:, user_idx])) % (2**16)
            if band_hash not in buckets:
                buckets[band_hash] = []
            buckets[band_hash].append(user_idx)

        # Add candidate pairs for columns hashed to the same bucket
        for bucket in buckets.values():
            if len(bucket) > 1:
                # Add all unique pairs from this bucket
                for i in range(len(bucket)):
                    for j in range(i + 1, len(bucket)):
                        candidate_pairs.add((bucket[i], bucket[j]))

    return candidate_pairs

# Vectorized Jaccard similarity calculation using sparse matrices
def jaccard_similarity(user1, user2, user_movie_matrix):
    user1_ratings = user_movie_matrix[user1].toarray().flatten()
    user2_ratings = user_movie_matrix[user2].toarray().flatten()
    intersection = np.dot(user1_ratings, user2_ratings)
    union = np.sum(np.logical_or(user1_ratings, user2_ratings))
    return intersection / union if union > 0 else 0.0

# Write results to a text file as u1,u2 pairs (where u1 < u2)
def write_results(candidate_pairs, user_movie_matrix, threshold=0.5):
    with open("result2.txt", "w") as file:  # Open file in write mode
        for u1, u2 in candidate_pairs:
            if u1 > u2:
                u1, u2 = u2, u1  # Ensure that u1 < u2
            similarity = jaccard_similarity(u1, u2, user_movie_matrix)
            if similarity > threshold:
                file.write(f"{u1+1},{u2+1}\n")  # Write to the file in the required format
    print(f"Results saved to result.txt")

# Main function to implement the LSH algorithm
def main():
    # Set random seed directly in the notebook for testing
    random_seed = 42  # You can change this value to any other seed if needed

    np.random.seed(random_seed)  # Set the random seed for numpy

    print(f"Starting the LSH algorithm with random seed {random_seed}...")  
    data = load_data('user_movie_rating.npy')  # Load data from .npy file
    if data is None:
        print("Error loading data, exiting.")
        return

    user_movie_matrix = create_sparse_matrix(data)  # Create sparse matrix
    num_hashes = 100  # Number of hash functions
    num_bands = 10  # Number of bands (you can experiment with different values)
    rows_per_band = num_hashes // num_bands  # Number of rows per band
    threshold = 0.3  # Threshold for similarity

    signature_matrix = minhash(user_movie_matrix, num_hashes)  # Generate signature matrix
    candidate_pairs = lsh_banding(signature_matrix, num_bands, rows_per_band, threshold)  # Apply LSH banding
    write_results(candidate_pairs, user_movie_matrix, threshold)  # Write results to text file

# Call the main function to execute the algorithm
main()


Starting the LSH algorithm with random seed 42...
Data loaded successfully with shape (65225506, 3)
Number of users: 103703, Number of movies: 17770
Sparse matrix shape: (103703, 17770)
